# Experiment 1 · build the questions (run locally)

200 MuSiQue 2-hop questions, each with its 2 supporting paragraphs and 16 distractor paragraphs, all turned into facts
by Llama-3.1-70B, the same as every earlier set. Output: `questions/MuSiQue_exp1.pkl`, read by `run_experiment_1.ipynb`.

**Sources.** First the MuSiQue validation split, then (only 73 fresh validation questions pass the filters) the train
split, using only questions the LoRA never saw: never tried during training-data generation and sharing no paragraph
with a training sample. Each question records its source (`source_split`).

Steps (the same filters as the test and dev sets, from `pipeline.py`):
1. **Pool**: 2-hop, shuffled with seed 0. Excluded: every test and dev question, any question whose supporting
   paragraph appears in the test set, the dev set or the LoRA training data, and (train split) every question tried
   during training-data generation.
2. **Closed book**: drop the questions the 70B answers without context (the facts must matter).
3. **Evidence**: drop the questions the 70B cannot answer from the 2 supporting paragraphs.
4. **Select and clean**: drop time-sensitive questions, repeated chains, and answers that are not one entity, date or number.
5. **Facts**: facts of the supporting paragraphs and of 16 distractor paragraphs per question.
6. **Layout**: each question's 16 distractors get a fixed order (k distractors = the first k, so the sets nest) and every
   paragraph a fixed reading position, so adding distractors only inserts paragraphs, never reorders the others.

Cost: about $1 of 70B calls.

In [1]:
import json, os, pickle, random
from collections import Counter

from llm import DeepInfra
from MuSiQue import load_pool
from pipeline import (CLOSED_BOOK, EVIDENCE, EXTRACT, answer_and_grade, parse_facts, removal_reason, select,
                      supporting)

N_QUESTIONS = 200
K_MAX = 16                 # distractor paragraphs per question
SEED = 0
TRAIN_BATCH = 400          # train-split candidates screened per round of the top-up
OUT = "questions/MuSiQue_exp1.pkl"
EXCLUDE = ["../runs/MuSiQue_test_entity.pkl", "../runs/MuSiQue_dev_entity.pkl"]
TRAINING = "../data/train/MuSiQue"   # the LoRA's training samples and the log of every question tried for them

llm = DeepInfra()
llm.check()

/Users/mina/miniforge3/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. Pools and exclusions
held = [q for path in EXCLUDE for q in pickle.load(open(path, "rb"))]
seen_ids = {q["id"] for q in held}
seen_texts = {p["text"] for q in held for p in q["supporting"]}
with open(f"{TRAINING}/MuSiQue_train.jsonl") as f:
    seen_texts |= {p["text"] for line in f for p in json.loads(line)["paragraphs"]}
with open(f"{TRAINING}/log.jsonl") as f:
    tried = {json.loads(line)["id"] for line in f if line.strip()}


def fresh(split):
    pool = [r for r in load_pool(SEED, split=split) if r["hops"] == 2]
    out = [r for r in pool if r["id"] not in seen_ids and r["id"] not in tried
           and not seen_texts & {p["text"] for p in r["paragraphs"] if p["is_supporting"]}]
    for r in out:
        r["supporting"], r["source_split"] = supporting(r), split
    print(f"1. {split}: {len(pool)} 2-hop questions, {len(out)} fresh")
    return out


pools = {"validation": fresh("validation"), "train": fresh("train")}

1. validation: 1252 2-hop questions, 440 fresh


1. train: 14376 2-hop questions, 6637 fresh


In [3]:
# 2-5. The same filters as every earlier set, then facts, until N_QUESTIONS questions are complete
def extract(paragraph_list):
    replies = llm.ask_all([EXTRACT.format(title=p["title"], text=p["text"]) for p in paragraph_list],
                          max_tokens=384, desc="Extracting facts")
    return [parse_facts(reply or "") for reply in replies]


def screen(candidates, need, counts):
    """Closed book, evidence, select and clean, facts: the complete questions among the candidates (at most need)."""
    answer_and_grade(llm, candidates, "closed_book", lambda r: CLOSED_BOOK.format(question=r["question"]))
    unknown = [r for r in candidates if r["closed_book"]["correct"] is False]
    paragraphs = lambda r: "\n\n".join(f"{p['title']}: {p['text']}" for p in r["supporting"])
    answer_and_grade(llm, unknown, "evidence", lambda r: EVIDENCE.format(paragraphs=paragraphs(r), question=r["question"]))
    answerable = [r for r in unknown if r["evidence"]["correct"] is True]
    selected = [r for r in select(answerable, per_hop=10 ** 6) if not removal_reason(r)][:need + 10]
    paragraph_list = [p for r in selected for p in r["paragraphs"]]
    for p, facts in zip(paragraph_list, extract(paragraph_list)):
        p["facts"] = facts
    complete = []
    for r in selected:
        for s in r["supporting"]:
            s["facts"] = next(p["facts"] for p in r["paragraphs"] if p["text"] == s["text"])
        if all(s["facts"] for s in r["supporting"]) and \
                sum(not p["is_supporting"] and bool(p["facts"]) for p in r["paragraphs"]) >= K_MAX:
            complete.append(r)
    for key, value in [("screened", len(candidates)), ("unknown", len(unknown)), ("answerable", len(answerable)),
                       ("selected", len(selected)), ("complete", len(complete))]:
        counts[key] += value
    return complete[:need]


kept, stats, seen_chains = [], {}, set()
for split, pool in pools.items():
    counts, start = Counter(), 0
    while len(kept) < N_QUESTIONS and start < len(pool):
        batch = pool if split == "validation" else pool[start:start + TRAIN_BATCH]
        start += len(batch)
        new = screen(batch, N_QUESTIONS - len(kept), counts)
        new = [r for r in new if frozenset(p["text"] for p in r["supporting"]) not in seen_chains]   # no repeated chain
        seen_chains |= {frozenset(p["text"] for p in r["supporting"]) for r in new}
        kept += new
        print(f"   {split}: {dict(counts)} → {len(kept)} questions so far")
    stats[split] = dict(counts)
print(f"5. {len(kept)} complete questions: {dict(Counter(r['source_split'] for r in kept))}")

closed_book: answering:   0%|          | 0/440 [00:00<?, ?it/s]

closed_book: answering:   0%|          | 1/440 [00:00<02:47,  2.63it/s]

closed_book: answering:   0%|          | 2/440 [00:01<04:39,  1.57it/s]

closed_book: answering:   1%|          | 5/440 [00:01<01:43,  4.20it/s]

closed_book: answering:   6%|▌         | 25/440 [00:01<00:16, 24.52it/s]

...

Extracting facts:  99%|█████████▉| 536/540 [02:36<00:00,  6.03it/s]

Extracting facts: 100%|█████████▉| 538/540 [02:36<00:00,  5.73it/s]

Extracting facts: 100%|██████████| 540/540 [02:37<00:00,  5.92it/s]

Extracting facts: 100%|██████████| 540/540 [02:37<00:00,  3.44it/s]

   train: {'screened': 800, 'unknown': 487, 'answerable': 396, 'selected': 136, 'complete': 136} → 200 questions so far
5. 200 complete questions: {'validation': 74, 'train': 126}


In [4]:
# 6. Layout: a fixed distractor order (nested sets) and a fixed reading position for every paragraph
for r in kept:
    rng = random.Random(r["id"])
    distractors = [p for p in r["paragraphs"] if not p["is_supporting"] and p["facts"]]
    rng.shuffle(distractors)
    layout = [{"title": p["title"], "text": p["text"], "facts": p["facts"], "is_supporting": True, "rank": None}
              for p in r["supporting"]]
    layout += [{"title": p["title"], "text": p["text"], "facts": p["facts"], "is_supporting": False, "rank": rank}
               for rank, p in enumerate(distractors[:K_MAX])]
    for p in layout:
        p["position"] = rng.random()                              # reading order: by position
    r["layout"] = sorted(layout, key=lambda p: p["position"])
    r.pop("paragraphs")


def paragraphs_at(q, k):
    """The paragraphs a system reads with k distractors: the 2 supporting ones and the first k distractors, in
    their fixed reading order (the same function is in run_experiment_1.ipynb)."""
    return [p for p in q["layout"] if p["is_supporting"] or p["rank"] < k]


for q in kept:
    assert all(set(map(id, paragraphs_at(q, a))) <= set(map(id, paragraphs_at(q, b))) for a, b in [(0, 4), (4, 8), (8, 16)])
    assert len(paragraphs_at(q, 0)) == 2 and len(paragraphs_at(q, K_MAX)) == 2 + K_MAX
assert len({q["id"] for q in kept}) == len(kept)
os.makedirs("questions", exist_ok=True)
pickle.dump(kept, open(OUT, "wb"))
stats.update(questions=len(kept), by_source=dict(Counter(r["source_split"] for r in kept)), api=llm.summary())
json.dump(stats, open("questions/MuSiQue_exp1_stats.json", "w"), indent=2)
q = kept[0]
print(f"6. Saved {len(kept)} questions to {OUT}. API: {llm.summary()}")
print(f"   facts per question (first): {sum(len(p['facts']) for p in q['layout'] if p['is_supporting'])} supporting, "
      f"{sum(len(p['facts']) for p in q['layout'] if not p['is_supporting'])} in 16 distractors")
print(f"   example: {q['question']}  →  {q['answer']}")

6. Saved 200 questions to questions/MuSiQue_exp1.pkl. API: {'model': 'meta-llama/Meta-Llama-3.1-70B-Instruct-Turbo', 'requests': 7260, 'failed': 0, 'input_tokens': 1286951, 'output_tokens': 689633, 'cost_usd': 0.7906}
   facts per question (first): 10 supporting, 121 in 16 distractors
   example: Who followed Menucha Rochel Slonim's father?  →  Menachem Mendel Schneersohn
